# JPEG encoding audit for the matched 822-image analysis

Run this notebook on the **actual transformed JPG files**. It inspects JPEG metadata, subsampling, and quantization-table signatures. JPEG files do not always record the encoder name; if it is absent, the encoder must be recovered from the original generation code rather than guessed.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
from pathlib import Path
JPG_ROOT = Path('/content/gdrive/MyDrive/2026-0/REPLACE_WITH_822_JPG_FOLDER')
OUTPUT_CSV = Path('/content/gdrive/MyDrive/2026-0/jpeg_encoding_audit.csv')
print(JPG_ROOT)


In [ ]:
import hashlib, json
import pandas as pd
import PIL
from PIL import Image, JpegImagePlugin, features

def qtable_signature(qtables):
    payload = json.dumps(qtables, sort_keys=True, default=list).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

rows = []
paths = sorted([p for p in JPG_ROOT.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg'}])
for path in paths:
    with Image.open(path) as im:
        exif = im.getexif()
        rows.append({
            'relative_path': str(path.relative_to(JPG_ROOT)),
            'width': im.width, 'height': im.height,
            'subsampling': JpegImagePlugin.get_sampling(im),
            'quantization_signature': qtable_signature(im.quantization or {}),
            'quantization_tables': len(im.quantization or {}),
            'comment': im.info.get('comment'),
            'software_exif': exif.get(305),
            'jfif_version': im.info.get('jfif_version'),
            'dpi': im.info.get('dpi'),
        })
audit = pd.DataFrame(rows)
audit.to_csv(OUTPUT_CSV, index=False)
print('Files:', len(audit))
print('Pillow used to inspect:', PIL.__version__)
print('Local JPEG decoder:', features.version_codec('jpg'))
display(audit.groupby(['quantization_signature', 'subsampling'], dropna=False).size().reset_index(name='n'))
display(audit[['comment', 'software_exif']].drop_duplicates())
print('Saved:', OUTPUT_CSV)


## Interpretation

- One quantization signature across all files indicates one consistent compression setting, not by itself a named quality factor.
- `software_exif` or `comment` may identify the encoder, but these fields are often empty.
- The definitive evidence remains the original code or a contemporaneous environment record.
- Do not report the Pillow decoder version printed here as the historical encoder used to create the files.
